In [2]:
#import libraries
import numpy as np
import pandas as pd
import RNA
import re
from scipy.stats import spearmanr
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.metrics import roc_curve, roc_auc_score


In [48]:
# RNU4-2 and RNU6 transcript sequences (5' to 3')
RNU4_2 = "AGCUUUGCGCAGUGGCAGUAUCGUAGCCAAUGAGGUUUAUCCGAGGCGCGAUUAUUGCUAAUUGAAAACUUUUCCCAAUACCCCGCCAUGACGACUUGAAAUAUAGUCGGCAUUGGCAAUUUUUGACAGUCUCUACGGAGACUGA"
RNU6   = "GUGCUCGCUUCGGCAGCACAUAUACUAAAAUUGGAACGAUACAGAGAAGAUUAGCAUGGCCCCUGCGCAAGGAUGACACGCAAAUUCGUGAAGCGUUCCAUAUUUU"

# 1) ViennaRNA helper funcs
def make_md(temp_c: float = 37.0):
    """
    Create a ViennaRNA 'model details' object (RNA.md) to control global
    folding parameters. Here we only set temperature; all other settings
    remain at ViennaRNA defaults (Turner parameters, etc.).
    """
    md = RNA.md()
    md.temperature = temp_c
    return md

# Reuse a single RNA.md object for all computations
md = make_md(37.0)

def mfe_single(seq: str) -> float:
    """
    Compute the minimum free energy (MFE, kcal/mol) of a single RNA strand.
    Returns the MFE value only; the structure is computed but not retained.
    """
    fc = RNA.fold_compound(seq, md) 
    struct, dG = fc.mfe()            
    return float(dG)

def mfe_complex(u4: str, u6: str) -> float:
    """
    Compute the MFE (kcal/mol) of the U4–U6 dimer using ViennaRNA cofold grammar.

    Implementation details:
      - ViennaRNA uses '&' to separate two interacting strands (cofold / dimer).
      - Prefer fc.mfe_dimer() (modern API) to ensure dimer-specific output.
      - Fall back to RNA.cofold() for older bindings if needed.
      - Worst-case fallback: fc.mfe() on the dimer string (still returns an MFE).

    Returns:
      dG (float): MFE of the dimer (kcal/mol)
    """
    dimer_seq = u4 + "&" + u6
    fc = RNA.fold_compound(dimer_seq, md)
    try:
        struct, dG = fc.mfe_dimer()        
    except Exception:
        if hasattr(RNA, "cofold"):
            struct, dG = RNA.cofold(dimer_seq)  
        else:
            struct, dG = fc.mfe()               
    return float(dG)

# 2) HGVS n.annotation parser

# Convert DNA-style HGVS alleles (T) to RNA (U) as they appear inHGVS annotation
_DNA2RNA = str.maketrans({"T": "U", "t": "u"})

def apply_hgvs_n(seq_rna: str, hgvs: str) -> str:
    """
    Apply HGVS "n." notation edits to an RNA sequence string.

    Input:
      seq_rna: RNA sequence (A/C/G/U)
      hgvs:    HGVS string starting with "n."

    Supported inputs:
      - Substitution: n.<pos><ref>><alt>         e.g. n.57C>U (or C>T, auto T->U)
      - Single del:   n.<pos>del                e.g. n.57del
      - Range del:    n.<a>_<b>del              e.g. n.57_60del (not needed in our dataset)
      - Insertion:    n.<a>_<b>ins<seq>         e.g. n.57_58insAUG
    """
    if not isinstance(hgvs, str) or not hgvs.startswith("n."):
        raise ValueError(f"Unsupported HGVS (expect 'n.'): {hgvs}")

    body = hgvs[2:]  # strip "n."

    # substitution: n.<pos><ref>><alt>
    m = re.fullmatch(r"(\d+)([ACGTU])>([ACGTU])", body, flags=re.I)
    if m:
        pos = int(m.group(1))  # HGVS is 1-based
        alt = m.group(3).upper().translate(_DNA2RNA)
        # replace base at pos with alt
        return seq_rna[:pos - 1] + alt + seq_rna[pos:]

    # single-base deletion: n.<pos>del
    m = re.fullmatch(r"(\d+)del", body, flags=re.I)
    if m:
        pos = int(m.group(1))
        # delete base at pos
        return seq_rna[:pos - 1] + seq_rna[pos:]

    # range deletion: n.<a>_<b>del
    m = re.fullmatch(r"(\d+)_(\d+)del", body, flags=re.I)
    if m:
        a, b = sorted((int(m.group(1)), int(m.group(2))))
        # delete bases from a..b inclusive
        return seq_rna[:a - 1] + seq_rna[b:]

    # insertion: n.<a>_<b>ins<seq>
    m = re.fullmatch(r"(\d+)_(\d+)ins([ACGTU]+)", body, flags=re.I)
    if m:
        a = int(m.group(1))
        ins = m.group(3).upper().translate(_DNA2RNA)
        # insert after position a (i.e., between a and a+1 in 1-based terms)
        return seq_rna[:a] + ins + seq_rna[a:]

    raise ValueError(f"Unsupported or unparsable HGVS: {hgvs}")


# 3) Core: compute ΔΔG terms for a list of HGVS variants
def compute_ddg_terms(hgvs_list):
    """
    Compute U4/U6 stability changes for a list of U4 variants.

    For each variant (HGVS in n.-notation applied to RNU4_2):
      - Fold WT U4+U6 dimer => G_complex_WT
      - Fold WT U4 alone    => G_U4_WT
      - Fold variant dimer  => G_complex_var
      - Fold variant U4     => G_U4_var

    Then compute:
      dG_complex = G_complex_var - G_complex_WT
      dG_U4      = G_U4_var      - G_U4_WT
      ddG_bind   = dG_complex    - dG_U4

    Returns:
      pandas.DataFrame with:
        - HGVS_expanded : the input HGVS string
        - ddG_bind_mfe  : ΔΔG_bind (kcal/mol), positive => destabilisation
    """

    # Precompute WT energies once (shared across all variants)
    G_complex_WT = mfe_complex(RNU4_2, RNU6)  # WT dimer MFE
    G_U4_WT      = mfe_single(RNU4_2)         # WT U4 single-strand MFE

    rows = []

    for hgvs in hgvs_list:
        # Apply variant to U4; if parsing fails, skip and continue
        try:
            u4_var = apply_hgvs_n(RNU4_2, hgvs)
        except Exception as e:
            print(f"[WARN] Skipping {hgvs}: {e}")
            continue

        # Fold variant dimer and variant U4 alone
        G_complex_var = mfe_complex(u4_var, RNU6)
        G_U4_var      = mfe_single(u4_var)

        # Δ terms relative to WT
        dG_complex = G_complex_var - G_complex_WT
        dG_U4      = G_U4_var      - G_U4_WT

        # ΔΔG_bind for U4/U6 (U6 constant => cancels)
        ddG_bind = dG_complex - dG_U4

        rows.append({
            "HGVS_expanded": hgvs,
            "ddG_bind_mfe":  ddG_bind,
        })

    return pd.DataFrame(rows)

In [49]:
#load function scores
df=pd.read_csv("RNU4_2_function_scores.csv")

In [50]:
#Calculate ddG values, Note: Unsupported HGVS is thrown for variants outside of the transcript
df_energies = compute_ddg_terms(df["HGVS_expanded"])

[WARN] Skipping nan: Unsupported HGVS (expect 'n.'): nan
[WARN] Skipping nan: Unsupported HGVS (expect 'n.'): nan
[WARN] Skipping nan: Unsupported HGVS (expect 'n.'): nan
[WARN] Skipping nan: Unsupported HGVS (expect 'n.'): nan
[WARN] Skipping nan: Unsupported HGVS (expect 'n.'): nan
[WARN] Skipping nan: Unsupported HGVS (expect 'n.'): nan
[WARN] Skipping nan: Unsupported HGVS (expect 'n.'): nan
[WARN] Skipping nan: Unsupported HGVS (expect 'n.'): nan
[WARN] Skipping nan: Unsupported HGVS (expect 'n.'): nan
[WARN] Skipping nan: Unsupported HGVS (expect 'n.'): nan
[WARN] Skipping nan: Unsupported HGVS (expect 'n.'): nan
[WARN] Skipping nan: Unsupported HGVS (expect 'n.'): nan
[WARN] Skipping nan: Unsupported HGVS (expect 'n.'): nan
[WARN] Skipping nan: Unsupported HGVS (expect 'n.'): nan
[WARN] Skipping nan: Unsupported HGVS (expect 'n.'): nan
[WARN] Skipping nan: Unsupported HGVS (expect 'n.'): nan
[WARN] Skipping nan: Unsupported HGVS (expect 'n.'): nan
[WARN] Skipping nan: Unsupporte

In [ ]:
#save data for plotting in main notebook
